In [82]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

Downloading Stop Words

In [119]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mshah\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [8]:
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [9]:
data_set = pd.read_csv("sentimentdataset3.csv")

In [10]:
df = pd.DataFrame(data_set)
print(df.head())

                                             Comment Sentiment
0  lets not forget that apple pay in 2014 require...   neutral
1  here in nz 50 of retailers don’t even have con...  negative
2  i will forever acknowledge this channel with t...  positive
3  whenever i go to a place that doesn’t take app...  negative
4  apple pay is so convenient secure and easy to ...  positive


No. of values in each Sentiment

In [11]:
df["Sentiment"] = df["Sentiment"].str.strip()
print(df["Sentiment"].value_counts())

Sentiment
positive    11432
neutral      4638
negative     2338
Name: count, dtype: int64


Finding Null Values

In [12]:
df.isnull().sum()

Comment      44
Sentiment     0
dtype: int64

Drop Null value Rows

In [13]:
df_cleaned = df.dropna()
df_cleaned.isnull().sum()

Comment      0
Sentiment    0
dtype: int64

In [14]:
print(df_cleaned["Sentiment"].value_counts())

Sentiment
positive    11402
neutral      4625
negative     2337
Name: count, dtype: int64


Replace the positive=1, negative=-1, neutral=0

In [15]:
df_cleaned.replace({"Sentiment":{"positive":1}}, inplace=True)
df_cleaned.replace({"Sentiment":{"neutral":0}}, inplace=True)
df_cleaned.replace({"Sentiment":{"negative":-1}}, inplace=True)
print(df_cleaned.head())

                                             Comment Sentiment
0  lets not forget that apple pay in 2014 require...         0
1  here in nz 50 of retailers don’t even have con...        -1
2  i will forever acknowledge this channel with t...         1
3  whenever i go to a place that doesn’t take app...        -1
4  apple pay is so convenient secure and easy to ...         1


In [17]:
ps = PorterStemmer()

In [18]:
def stemming(content):
    sc = re.sub('[^a-zA-Z]',' ',content)
    sc = sc.lower()
    sc = sc.split()
    sc = [ps.stem(word) for word in sc if not word in stopwords.words("english")]
    sc = " ".join(sc)

    return sc

In [19]:
df_cleaned['stemmed_comments'] = df_cleaned["Comment"].apply(stemming)
print(df_cleaned.head())

                                             Comment Sentiment  \
0  lets not forget that apple pay in 2014 require...         0   
1  here in nz 50 of retailers don’t even have con...        -1   
2  i will forever acknowledge this channel with t...         1   
3  whenever i go to a place that doesn’t take app...        -1   
4  apple pay is so convenient secure and easy to ...         1   

                                    stemmed_comments  
0  let forget appl pay requir brand new iphon ord...  
1  nz retail even contactless credit card machin ...  
2  forev acknowledg channel help lesson idea expl...  
3  whenev go place take appl pay happen often dra...  
4  appl pay conveni secur easi use use korean jap...  


In [20]:
df_cleaned["stemmed_comments"]

0        let forget appl pay requir brand new iphon ord...
1        nz retail even contactless credit card machin ...
2        forev acknowledg channel help lesson idea expl...
3        whenev go place take appl pay happen often dra...
4        appl pay conveni secur easi use use korean jap...
                               ...                        
18403    realli like point engin toolbox think lot burn...
18404    start explor field realli good remind get earl...
18405    excelent video con una pregunta filo fica prof...
18406    hey daniel discov channel coupl day ago im lea...
18407    great focu key play approach also speed thing ...
Name: stemmed_comments, Length: 18364, dtype: str

Spliting into Test And Train

In [21]:
text = df_cleaned["stemmed_comments"].values
labels = df_cleaned["Sentiment"].values

In [125]:
print(labels)

[0 -1 1 ... 0 1 1]


In [85]:
X_train, X_test, Y_train, Y_test = train_test_split(
    text, labels, 
    test_size=0.2,   # 20% test, 80% train
    random_state=2
)

In [86]:
print(text.shape, X_train.shape, X_test.shape)

(18364,) (14691,) (3673,)


In [87]:
print(labels.shape, Y_train.shape, Y_test.shape)


(18364,) (14691,) (3673,)


Converting text data into numeric form

In [88]:
vector = TfidfVectorizer()
X_train = vector.fit_transform(X_train)
X_test = vector.transform(X_test)


In [89]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 225479 stored elements and shape (14691, 21324)>
  Coords	Values
  (0, 7678)	0.478392754548417
  (0, 13258)	0.4364957728262302
  (0, 338)	0.7619788794319762
  (1, 13258)	0.12675855287948076
  (1, 1799)	0.1567896032074244
  (1, 18824)	0.15115065212326367
  (1, 5438)	0.2182292070745934
  (1, 12666)	0.16191080311384223
  (1, 18412)	0.15889189278382393
  (1, 4084)	0.23969842692911725
  (1, 150)	0.4653318429978797
  (1, 17424)	0.17578833754311016
  (1, 572)	0.30311198936018974
  (1, 15684)	0.19941612911883463
  (1, 19887)	0.16258518958613016
  (1, 1232)	0.24523012878328287
  (1, 18550)	0.20604228159025378
  (1, 14331)	0.24705936841892842
  (1, 20181)	0.11421644417975345
  (1, 3727)	0.24181574056183264
  (1, 3536)	0.32668213389390427
  (1, 6255)	0.18049319313386966
  (2, 7998)	0.10681492066213012
  (2, 16572)	0.18115518790097612
  (2, 3867)	0.12599473092311717
  :	:
  (14687, 6245)	0.17569369549215672
  (14687, 17352)	0.2914206808

In [90]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 54897 stored elements and shape (3673, 21324)>
  Coords	Values
  (0, 198)	0.3003786461931743
  (0, 1302)	0.42042596483434086
  (0, 2360)	0.3003786461931743
  (0, 3006)	0.2875167807665442
  (0, 10641)	0.22513530606402088
  (0, 10678)	0.1535583614795972
  (0, 10882)	0.1938075871613116
  (0, 15299)	0.2620212862428732
  (0, 15304)	0.36731487372599636
  (0, 15311)	0.17937510382323335
  (0, 15919)	0.4610240673312289
  (1, 7133)	0.7584392384731727
  (1, 7841)	0.5009596847824986
  (1, 20181)	0.4169044444076525
  (2, 8233)	0.17052859365192183
  (2, 10139)	0.16890867157804956
  (2, 12701)	0.5895782836342361
  (2, 12720)	0.31289951038903124
  (2, 16955)	0.3280696771302827
  (2, 18740)	0.13447972646703413
  (2, 21016)	0.6092345142841408
  (3, 892)	0.25083657453862884
  (3, 3624)	0.22743066950438068
  (3, 3673)	0.21347304688816843
  (3, 3803)	0.3056289613066577
  :	:
  (3671, 7734)	0.33762836936847357
  (3671, 9191)	0.4297856146818536
  

Training ML Model Linear Logistics

In [91]:
model = LogisticRegression(max_iter=1000)

In [92]:
print(pd.DataFrame(Y_train).tail())
Y_train = Y_train.astype(int)

        0
14686   1
14687  -1
14688   0
14689  -1
14690   1


In [93]:
model.fit(X_train, Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [94]:
X_train_pred = model.predict(X_train)

In [95]:
# Accuracy Score on traning data
print("Accuracy:", accuracy_score(Y_train, X_train_pred))

Accuracy: 0.8783609012320468


In [97]:
# Accuracy Score on test data
X_test_pred = model.predict(X_test)
Y_test = Y_test.astype(int)
print("Accuracy:", accuracy_score(Y_test, X_test_pred))


Accuracy: 0.755785461475633


THE ACCURACY ON TEST DATA = 75.5%
THE ACCURACY ON TRAIN DATA = 87.8%

In [98]:
import pickle

In [99]:
# Save model
pickle.dump(model, open('model.pkl', 'wb'))

# Save vectorizer
pickle.dump(vector, open('vectorizer.pkl', 'wb'))